In [2]:
import pandas as pd
import numpy as np

In [3]:
# 시각을 변환하고 요일 편향을 확인.
orders = pd.read_csv("../data/orders.csv")

orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   order_id        200000 non-null  int64
 1   customer_id     200000 non-null  int64
 2   order_datetime  198067 non-null  str  
 3   channel         200000 non-null  str  
 4   status          200000 non-null  str  
dtypes: int64(2), str(3)
memory usage: 13.4 MB


In [4]:
print(f"order_datetime 변환 전 데이터 타입 : {orders['order_datetime'].dtype}")

order_datetime 변환 전 데이터 타입 : str


In [5]:
# 변환
orders["order_datetime"] = pd.to_datetime(
    orders["order_datetime"], errors="coerce"   # errors default는 raise. 하나 안 되면 전체가 변환 안 됨.
)

In [6]:
# "2026-07-24 14:24:50" => 정상 변환 가능
# "206-07-24 14:24:50" => 정상 변환 불가능 -> 에러 -> NaT errors="coerce"
print(f"order_datetime 변환 후 데이터 타입: {orders['order_datetime'].dtype}")

order_datetime 변환 후 데이터 타입: datetime64[us]


In [7]:
# NaT(==NaN) 확인
orders["order_datetime"].isna().sum()

np.int64(1933)

In [8]:
# 1. 2024년 6월 이후의 주문 건수
june_on = orders[orders["order_datetime"] >= "2024-06-01"]
june_on.shape[0]

42959

In [9]:
print(f"최초 주문 : {orders['order_datetime'].min()}")
print(f"최후 주문 : {orders['order_datetime'].max()}")

최초 주문 : 2024-01-01 00:02:25
최후 주문 : 2025-06-30 03:01:14


In [10]:
# dt 접근자 : 시간 축 추출, 요일별 주문 개수
orders_df = orders.dropna(subset="order_datetime")  # 삭제

In [11]:
# 추출
orders_df["month"] = orders_df["order_datetime"].dt.month
orders_df["dow"] = orders_df["order_datetime"].dt.dayofweek
orders_df["dow_name"] = orders_df["order_datetime"].dt.day_name()

In [12]:
orders_df.head()

,order_id,customer_id,order_datetime,channel,status,month,dow,dow_name
0,25463,1077,2024-06-25 00:17:41,store,delivered,6,1,Tuesday
1,171088,2998,2024-05-28 19:35:20,web,delivered,5,1,Tuesday
2,27437,4066,2024-02-14 17:49:14,app,delivered,2,2,Wednesday
3,98425,3243,2024-05-08 16:37:36,store,delivered,5,2,Wednesday
4,22325,5331,2024-04-17 20:08:22,app,delivered,4,2,Wednesday


In [13]:
# 요일별 주문 건수
# orders_df.groupby("dow_name") DataFrameGroupBy 
# orders_df.groupby("dow_name")["order_id"]   SeriesGroupBy 
# orders_df.groupby("dow_name")["order_id"].count()
# 요일명으로 정렬 사전작업
order_names = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
orders_df.groupby("dow_name")["order_id"].count().reindex(order_names)

dow_name
Monday       25286
Tuesday      25134
Wednesday    24977
Thursday     25568
Friday       25478
Saturday     36037
Sunday       35587
Name: order_id, dtype: int64

In [14]:
# 일별 평균 환산
days = orders_df["order_datetime"].dt.normalize()     # 연-월-일만 나옴.(시:분:초가 00:00:00가 됨.)

In [15]:
day_count = orders_df.groupby(days).size()
day_count

order_datetime
2024-01-01    639
2024-01-02    687
2024-01-03    696
2024-01-04    677
2024-01-05    702
             ... 
2025-06-20      1
2025-06-22      1
2025-06-23      1
2025-06-24      1
2025-06-30      1
Length: 222, dtype: int64

In [16]:
# 주말 건수
is_weekend = day_count.index.dayofweek >= 5

In [17]:
# 주말 평균 주문 건수 : 1119
day_count[is_weekend].mean()

np.float64(1119.125)

In [18]:
# 평일 평균 주문 건수 : 800
day_count[~is_weekend].mean()

np.float64(800.2721518987341)

In [19]:
# 1. 시간대별 주문 건수: 언제(시간 단위) 가장 많이 주문하나?
orders_df["hour"] = orders_df["order_datetime"].dt.hour
# orders_df["hour"][:10]
by_hour = orders_df.groupby("hour")["order_id"].count()
by_hour

hour
0     8239
1     8239
2     8210
3     8216
4     8318
5     8157
6     8109
7     8355
8     8264
9     8182
10    8319
11    8264
12    8403
13    8326
14    8285
15    8278
16    8102
17    8309
18    8243
19    8212
20    8358
21    8235
22    8156
23    8288
Name: order_id, dtype: int64

In [20]:
# 가장 많은 건수 : max
print(f"가장 많은 건수 : {by_hour.max()}")
# 가장 많은 건수 있는 시간대
print(f"가장 많은 건수 있는 시간대 : {by_hour.idxmax()}시")

가장 많은 건수 : 8403
가장 많은 건수 있는 시간대 : 12시


In [21]:
# 가입일(customers)부터 첫 주문(orders_df)까지 걸린 일 수
customers_df = pd.read_csv("../data/customers.csv")
# customers_df.info()
# signup_date컬럼의 데이터 타입을 datetime 변경
customers_df["signup_date"] = pd.to_datetime(
    customers_df["signup_date"], errors="coerce"
)
# customers_df["customer_id"]
# orders_df["customer_id"].max()
# 고객별 최초 주문 날짜
first_order = orders_df.groupby("customer_id")["order_datetime"].min()
first_order

customer_id
1000     2024-01-21 23:31:35
1001     2024-01-03 12:01:52
1002     2024-01-06 10:37:15
1003     2024-01-03 09:42:56
1004     2024-01-01 04:23:36
                 ...        
989772   2024-01-10 04:09:52
989922   2024-06-29 03:27:02
989960   2024-06-09 16:13:18
989977   2024-01-14 00:14:02
989995   2024-05-27 01:54:32
Name: order_datetime, Length: 5941, dtype: datetime64[us]

In [23]:
cust = customers_df.drop_duplicates("customer_id").set_index("customer_id")
cust.head()

,name,gender,birth_date,signup_date,city,email
customer_id,,,,,,
3293,권준서,여,1954-04-29,2020-04-26,부산,user3293@example.com
1862,박준선,F,1980-05-19,2023-09-01,서울,user1862@example.com
4955,전영경,여,1980-06-08,2024-04-30,인천,user4955@example.com
4653,한재하,M,1971-06-09,2023-08-06,서울,user4653@example.com
3331,송준연,여,1971-02-01,2023-12-20,대전,user3331@example.com


In [ ]:
# first_order - cust['signup_date']     # 같은 index(customer_id)끼리 꺼내서 빼줌. 그래서 위 셀에 set_index("customer_id") 사용.
first_order.head(1)[:1], cust["signup_date"][:1]        

(customer_id
 1000   2024-01-21 23:31:35
 Name: order_datetime, dtype: datetime64[us],
 customer_id
 3293   2020-04-26
 Name: signup_date, dtype: datetime64[us])

In [ ]:
gap = (first_order-cust["signup_date"]).dt.days.dropna()
gap.mean()      # 가입 후 최초 주문 평균 일 수

np.float64(663.6129292929293)